# Task 2 (Advanced) – Support Vector Machine (SVM)

**Goal:** Binary classification of breast tumours (Malignant vs Benign) using SVMs with different kernels, and visualise the decision boundary.

**Why SVMs for medical diagnosis?**  
SVMs find the *maximum-margin* hyperplane between classes. That margin is like a safety buffer — especially valuable when a wrong prediction (misclassifying a malignant tumour as benign) has serious consequences.

**Why only 2 features?**  
We deliberately restrict to the first two features (Mean Radius, Mean Texture) so we can plot the decision boundary in 2D. In practice you'd use all features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

## 1. Load & Prepare Data

In [ ]:
cancer = datasets.load_breast_cancer()

# First two features only — for decision-boundary visualisation
X = cancer.data[:, :2]   # Mean Radius, Mean Texture
y = cancer.target

print("Feature names used:", cancer.feature_names[:2])
print(f"Class distribution: {dict(zip(cancer.target_names, np.bincount(y)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 2. Scale Features

SVMs are sensitive to feature scale — unscaled features can make the optimisation converge slowly or find a bad hyperplane. StandardScaler gives each feature zero mean and unit variance.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test  = scaler.transform(X_test)        # apply same transform to test data

## 3. Compare Kernels

- **Linear kernel**: good when classes are roughly linearly separable  
- **RBF kernel**: maps data into higher dimensions to find non-linear boundaries  

We compare both on accuracy and AUC (area under ROC curve).

In [ ]:
print(f"{'Kernel':<10} {'Accuracy':>10} {'AUC-ROC':>10}")
print("-" * 32)

results = {}
for kernel in ['linear', 'rbf']:
    svm = SVC(kernel=kernel, probability=True, random_state=42)
    svm.fit(X_train, y_train)
    y_pred  = svm.predict(X_test)
    y_proba = svm.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    results[kernel] = svm
    
    print(f"{kernel:<10} {acc:>10.4f} {auc:>10.4f}")

print("\nFull report for RBF kernel:")
print(classification_report(y_test, results['rbf'].predict(X_test),
                             target_names=cancer.target_names))

## 4. Visualise the Decision Boundary (RBF)

We create a fine mesh across the feature space and colour each point by its predicted class to show where the boundary sits.

In [ ]:
best_svm = results['rbf']

# Build the mesh grid
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                      np.arange(y_min, y_max, 0.02))

Z = best_svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.7, cmap=plt.cm.coolwarm)
scatter = plt.scatter(
    X_train[:, 0], X_train[:, 1],
    c=y_train, edgecolors='k', cmap=plt.cm.coolwarm, s=40
)
plt.colorbar(scatter, ticks=[0, 1], label='Class (0=Malignant, 1=Benign)')
plt.title("SVM Decision Boundary – RBF Kernel (Breast Cancer)")
plt.xlabel(f"Feature 1: {cancer.feature_names[0]} (scaled)")
plt.ylabel(f"Feature 2: {cancer.feature_names[1]} (scaled)")
plt.tight_layout()
plt.show()